# ACE unified configuration and environment setup

Run this notebook first in every new VS Code kernel or Colab runtime. It mounts Google My Drive when needed, validates the shared `ace_config.py` module, mirrors it into the active Colab session, creates the complete persistent directory registry, and records environment metadata. Raw data are never overwritten. CHEMBL1808 is training-only; CHEMBL2096989 is external-validation-only.

In [5]:
from pathlib import Path
import os, sys

# Resolve the one persistent project location.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IS_COLAB = True
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    IS_COLAB = False
    # VS Code: open the Google Drive for desktop-synced ACE_Workflow folder.
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
    if not (ROOT / 'ace_config.py').exists():
        ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
print(f'ACE workflow root: {ROOT}')
print(f'Runtime: {"Colab CPU" if IS_COLAB else "VS Code CPU"}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ACE workflow root: /content/drive/MyDrive/ACE_Workflow
Runtime: Colab CPU


In [6]:
# Validate the single shared configuration module, then make it available to this runtime.
# Keeping a verified copy in /content avoids import-path issues in Colab without creating result files outside My Drive.
import hashlib, py_compile, shutil
CONFIG_SOURCE = ROOT / 'ace_config.py'
if not CONFIG_SOURCE.exists():
    raise FileNotFoundError(f'Missing {CONFIG_SOURCE}. Upload the complete ACE_Workflow folder, including ace_config.py.')
SESSION_CONFIG_DIR = Path('/content') if IS_COLAB else ROOT
SESSION_CONFIG = SESSION_CONFIG_DIR / 'ace_config.py'
if SESSION_CONFIG.resolve() != CONFIG_SOURCE.resolve():
    shutil.copy2(CONFIG_SOURCE, SESSION_CONFIG)
py_compile.compile(str(CONFIG_SOURCE), doraise=True)
CONFIG_SHA256 = hashlib.sha256(CONFIG_SOURCE.read_bytes()).hexdigest()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(SESSION_CONFIG_DIR) not in sys.path: sys.path.insert(0, str(SESSION_CONFIG_DIR))
sys.modules.pop('ace_config', None)
from ace_config import CONFIG, PATHS, INPUTS, IC50_CUTOFF_NM, CPU_WORKERS, setup_workflow, verify_inputs, verify_paths, write_run_manifest
setup_workflow()
print(f'ace_config.py SHA256: {CONFIG_SHA256}')
print(f'CPU workers: {CPU_WORKERS}; default IC50 activity cutoff: {IC50_CUTOFF_NM:g} nM')
verify_paths(PATHS)
write_run_manifest(PATHS['DATA_PROCESSED'] / 'ace_workflow_environment.json', {'config_sha256': CONFIG_SHA256})

Environment : COLAB_CPU
Project root: /content/drive/MyDrive/ACE_Workflow
Directories : 29
ace_config.py SHA256: 2b4a735d48a99071feb9a71406f257a4e3c0862d30836d79f9c47ca0b851c5a0
CPU workers: 2; default IC50 activity cutoff: 25 nM
PROJECT                  OK      /content/drive/MyDrive/ACE_Workflow
NOTEBOOKS                OK      /content/drive/MyDrive/ACE_Workflow/notebooks
DATA                     OK      /content/drive/MyDrive/ACE_Workflow/data
DATA_RAW                 OK      /content/drive/MyDrive/ACE_Workflow/data/raw
DATA_PROCESSED           OK      /content/drive/MyDrive/ACE_Workflow/data/processed
ML                       OK      /content/drive/MyDrive/ACE_Workflow/ml_model
ML_RESULTS               OK      /content/drive/MyDrive/ACE_Workflow/ml_model/results
ML_CLASSIFICATION        OK      /content/drive/MyDrive/ACE_Workflow/ml_model/results/ic50_classifier
ML_REGRESSION            OK      /content/drive/MyDrive/ACE_Workflow/ml_model/results/pchembl_regressor
ML_MODELS       

PosixPath('/content/drive/MyDrive/ACE_Workflow/data/processed/ace_workflow_environment.json')

In [7]:
# Colab CPU only: set to True once per fresh Colab runtime, then run this cell.
# Packages are temporary; every workflow file remains in Google My Drive.
INSTALL_COLAB_CPU_DEPENDENCIES = False
if IS_COLAB and INSTALL_COLAB_CPU_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')])
    # Use the official Vina 1.2.5 release binary (distro packages can be much older).
    import urllib.request, os as _os
    _url = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64'
    urllib.request.urlretrieve(_url, '/usr/local/bin/vina'); _os.chmod('/usr/local/bin/vina', 0o755)
    print(subprocess.run(['vina', '--version'], text=True, capture_output=True).stdout.strip())
    print('Colab CPU dependencies installed.')

In [8]:
# Every required input must be in Google My Drive/ACE_Workflow/data/raw.
# CHEMBL1808/CHEMBL2096989 may be CSV, TSV, or XLSX; the receptor is 1o86.pdb.
verify_inputs()

CHEMBL1808           OK      /content/drive/MyDrive/ACE_Workflow/data/raw/CHEMBL1808.csv
CHEMBL2096989        OK      /content/drive/MyDrive/ACE_Workflow/data/raw/CHEMBL2096989.csv
FDA_DRUGBANK         OK      /content/drive/MyDrive/ACE_Workflow/data/raw/FDA_DrugBank_cleaned.csv
EXPERIMENTAL_ZIAI    OK      /content/drive/MyDrive/ACE_Workflow/data/raw/Experimental_ziai_cleaned.csv
NPASS                OK      /content/drive/MyDrive/ACE_Workflow/data/raw/NPASS_cleaned.csv
PLANTS_CMAUP         OK      /content/drive/MyDrive/ACE_Workflow/data/raw/Plants_CMAUP_cleaned.csv
GLAXO_ALERTS         OK      /content/drive/MyDrive/ACE_Workflow/data/raw/glaxo_structural_alerts.csv
RECEPTOR_1O86        OK      /content/drive/MyDrive/ACE_Workflow/data/raw/1o86.pdb


True

## Run order

1. `1-ACE-MODEL-IC50-CUTOFF.ipynb` (binary activity model)
2. `2-ACE-MODEL-REGRESSOR.ipynb` (DDR1-style pChEMBL regression)
3. `3-ACE-DRUGABILITY.ipynb` (ADMET + Glaxo structural alerts)
4. `4-ACE-REDOCK-1O86.ipynb`, then `5-ACE-DOCK-1O86.ipynb`, then `6-ACE-VIZ.ipynb`.

Do not use external-validation metrics to select the final model or tune thresholds.